In [1]:
import requests 
import pandas as pd
import openpyxl
import time
from openpyxl import Workbook
import schedule
import logging

In [3]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
## Step1 fetching data 
def fetch_crypto_data():    
    url="https://api.coingecko.com/api/v3/coins/markets"
    params={
        "vs_currency":'Usd',
        "order":'market_cap',
        'per_page':50,
        'page':1,
        'sparkline':'false'
    }
    response=requests.get(url,params=params,timeout=15)
    data=response.json()
    if data :
        logging.info('Fetching data successfully..!')
    else:
        logging.warning('Error Fetching data..! Api data Is Empty ')

    return data
crypto_data=fetch_crypto_data()
def process_crypto_data(crypto_data):
    if crypto_data is None:
        logging.warning("No data to process.")
        return None

crypto_list=[]
for coin in crypto_data:
    crypto_list.append([
        coin['name'],
        coin['symbol'].upper(),
        coin['current_price'],
        coin['market_cap'],
        coin['total_volume'],
        coin['price_change_percentage_24h']
    ])


2025-02-12 15:58:10,948 - INFO - Fetching data successfully..!


In [4]:
#df=fetch_crypto_data()
df=pd.DataFrame(crypto_list,columns=["Name", "Symbol", "Price (USD)", "Market Cap", "24h Volume", "24h Price Change (%)"])
print(df)


                 Name  Symbol   Price (USD)     Market Cap   24h Volume  \
0             Bitcoin     BTC  96040.000000  1905528543545  37004476886   
1            Ethereum     ETH   2622.960000   316520236861  20090029671   
2              Tether    USDT      0.999961   141947045747  65915034582   
3                 XRP     XRP      2.410000   139424103400   3909686600   
4              Solana     SOL    196.110000    95831357628   3516735344   
5                 BNB     BNB    648.730000    94475598783   1268906057   
6                USDC    USDC      0.999929    56037834366   7800747057   
7            Dogecoin    DOGE      0.254139    37656577622   1345677793   
8             Cardano     ADA      0.785160    28184134588   1283168234   
9   Lido Staked Ether   STETH   2620.830000    24787083612    112815683   
10               TRON     TRX      0.242214    20877090527    717034894   
11    Wrapped Bitcoin    WBTC  96018.000000    12397413295    417302839   
12          Chainlink    

In [5]:
# savaing data to excel
def save_to_excel(df,filename='Crypto_data.xlsx'):
    try:
        if df is not None:
            df.to_excel(filename,index=False)
            print('Data is successfully saved to file..!')
        else:
            print('Data is not saved.!!')
            return None
    except Exception as e:
        print('Error may occuring while saving file.Please check once again ..!!',e)
        return None
    return df

df1=save_to_excel(df)
print(df1)


Data is successfully saved to file..!
                 Name  Symbol   Price (USD)     Market Cap   24h Volume  \
0             Bitcoin     BTC  96040.000000  1905528543545  37004476886   
1            Ethereum     ETH   2622.960000   316520236861  20090029671   
2              Tether    USDT      0.999961   141947045747  65915034582   
3                 XRP     XRP      2.410000   139424103400   3909686600   
4              Solana     SOL    196.110000    95831357628   3516735344   
5                 BNB     BNB    648.730000    94475598783   1268906057   
6                USDC    USDC      0.999929    56037834366   7800747057   
7            Dogecoin    DOGE      0.254139    37656577622   1345677793   
8             Cardano     ADA      0.785160    28184134588   1283168234   
9   Lido Staked Ether   STETH   2620.830000    24787083612    112815683   
10               TRON     TRX      0.242214    20877090527    717034894   
11    Wrapped Bitcoin    WBTC  96018.000000    12397413295    

In [6]:
df1.head(5)

,Name,Symbol,Price (USD),Market Cap,24h Volume,24h Price Change (%)
0,Bitcoin,BTC,96040.000000,1905528543545,37004476886,-2.12901
1,Ethereum,ETH,2622.960000,316520236861,20090029671,-2.98165
2,Tether,USDT,0.999961,141947045747,65915034582,-0.01850
3,XRP,XRP,2.410000,139424103400,3909686600,-3.25935
4,Solana,SOL,196.110000,95831357628,3516735344,-3.16355


In [7]:
## Step2:  Data Anylsis
# a.Top 5 Cryptocurrencies by Market Cap
df3=df1.sort_values(by='Market Cap',ascending=False).head(5)
df3

,Name,Symbol,Price (USD),Market Cap,24h Volume,24h Price Change (%)
0,Bitcoin,BTC,96040.000000,1905528543545,37004476886,-2.12901
1,Ethereum,ETH,2622.960000,316520236861,20090029671,-2.98165
2,Tether,USDT,0.999961,141947045747,65915034582,-0.01850
3,XRP,XRP,2.410000,139424103400,3909686600,-3.25935
4,Solana,SOL,196.110000,95831357628,3516735344,-3.16355


In [8]:
# b.Average Price of the Top 50 Cryptocurrencies
avg_price=df['Price (USD)'].mean()
avg_price

4165.299308608601

In [9]:
# c.Highest and Lowest 24-Hour Percentage Price Change
highest=df.loc[df["24h Price Change (%)"].idxmax()]
highest

Name                            BNB
Symbol                          BNB
Price (USD)                  648.73
Market Cap              94475598783
24h Volume               1268906057
24h Price Change (%)        1.26251
Name: 5, dtype: object

In [10]:
Lowest=df.loc[df["24h Price Change (%)"].idxmin()]
Lowest

Name                      Litecoin
Symbol                         LTC
Price (USD)                 116.78
Market Cap              8834554264
24h Volume              1372236081
24h Price Change (%)      -8.42517
Name: 20, dtype: object

In [11]:
def update_crypto_data():
    print(" Fetching live crypto data...")
    crypto_data = fetch_crypto_data()
    df = process_crypto_data(crypto_data)
    save_to_excel(df)

schedule.every(1).minutes.do(update_crypto_data)

print("Live crypto data updates started... ..!")

while True:
    schedule.run_pending()
    time.sleep(1)  


Live crypto data updates started... ..!
 Fetching live crypto data...


2025-02-12 16:00:14,342 - INFO - Fetching data successfully..!


Data is not saved.!!


2025-02-12 16:01:14,491 - INFO - Fetching data successfully..!


 Fetching live crypto data...
Data is not saved.!!


2025-02-12 16:02:14,671 - INFO - Fetching data successfully..!


 Fetching live crypto data...
Data is not saved.!!


2025-02-12 16:03:14,830 - INFO - Fetching data successfully..!


 Fetching live crypto data...
Data is not saved.!!


KeyboardInterrupt: 